# Reddit — Preprocessing

Processes raw JSONL files into two parquet files:
- **`reddit_clean.parquet`** — comments from all 5 subreddits (incl. r/politics), with id/parent_id/link_id for network analysis
- **`reddit_posts.parquet`** — posts from all 5 subreddits (title, selftext, metadata)


In [1]:
import json
import re
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

DATA_DIR = Path('../../Data/Reddit')

SUBREDDITS = {
    'conservative': DATA_DIR / 'r_conservative_comments.jsonl',
    'worldnews':    DATA_DIR / 'r_worldnews_comments.jsonl',
    'democrats':    DATA_DIR / 'r_democrats_comments.jsonl',
    'trump':        DATA_DIR / 'r_trump_comments.jsonl',
    'republican':   DATA_DIR / 'r_republican_comments.jsonl',
    'liberal':      DATA_DIR / 'r_liberal_comments.jsonl',
    'politics':     DATA_DIR / 'r_politics_comments.jsonl',
}

POST_FILES = {
    'conservative': DATA_DIR / 'r_conservative_posts.jsonl',
    'worldnews':    DATA_DIR / 'r_worldnews_posts.jsonl',
    'democrats':    DATA_DIR / 'r_democrats_posts.jsonl',
    'trump':        DATA_DIR / 'r_trump_posts.jsonl',
    'republican':   DATA_DIR / 'r_republican_posts.jsonl',
    'liberal':      DATA_DIR / 'r_liberal_posts.jsonl',
    'politics':     DATA_DIR / 'r_politics_posts.jsonl',
}

## 1. Helper functions

In [2]:
def clean_text(text: str) -> str:
    """Basic cleaning: lowercase, remove URLs, markdown links, Reddit mentions,
    HTML entities and extra whitespace."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\[([^\]]+)\]\(https?://[^)]+\)', r'\1', text)   # markdown links
    text = re.sub(r'https?://\S+|www\.\S+', '', text)                 # URLs
    text = re.sub(r'/u/\w+|/r/\w+', '', text)                         # Reddit mentions
    text = re.sub(r'&amp;|&lt;|&gt;|&quot;|&#\d+;', ' ', text)        # HTML entities
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_comments(file_path, subreddit_name, min_words=3, max_words=500):
    """Reads a comments JSONL file and returns a list of dicts."""
    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            body = obj.get('body', '')
            if not isinstance(body, str):
                continue
            if body.lower().strip() in ('[deleted]', '[removed]', ''):
                continue

            created_utc = obj.get('created_utc')
            if created_utc is None:
                continue

            text_clean = clean_text(body)
            if not text_clean:
                continue

            word_count = len(text_clean.split())
            if not (min_words <= word_count <= max_words):
                continue

            sentiment = sia.polarity_scores(text_clean)['compound']
            body_clean = re.sub(r'https?://\S+|www\.\S+', '', body)  # body with URL-only cleaning

            rows.append({
                'id':             obj.get('id', ''),
                'parent_id':      obj.get('parent_id', ''),
                'link_id':        obj.get('link_id', ''),
                'subreddit':      subreddit_name,
                'created_utc':    int(created_utc),
                'datetime':       pd.to_datetime(created_utc, unit='s', utc=True),
                'date':           pd.to_datetime(created_utc, unit='s', utc=True).date(),
                'author':         obj.get('author', ''),
                'score':          obj.get('score', 0),
                'body':           body,
                'body_clean':     body_clean,
                'text_clean':     text_clean,
                'length_chars':   len(text_clean),
                'length_words':   word_count,
                'sentiment':      sentiment,
                'mentions_trump':  int('trump' in text_clean),
                'mentions_harris': int('harris' in text_clean or 'kamala' in text_clean),
                'mentions_biden':  int('biden' in text_clean),
            })

            if i % 200_000 == 0:
                print(f'  {subreddit_name}: {i:,} lines processed, {len(rows):,} kept...')

    return rows

## 2. Load & combine all comments

In [3]:
all_rows = []

for name, path in SUBREDDITS.items():
    print(f'
Loading: r/{name} ({path.name})')
    rows = parse_comments(path, name)
    print(f'  → {len(rows):,} comments kept')
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print(f'\nTotal before deduplication: {df.shape}')


Inladen: r/conservative (r_conservative_comments.jsonl)
  conservative: 600,000 regels verwerkt, 268,083 bewaard...
  → 302,936 comments bewaard

Inladen: r/worldnews (r_worldnews_comments.jsonl)
  worldnews: 200,000 regels verwerkt, 165,566 bewaard...
  worldnews: 400,000 regels verwerkt, 331,149 bewaard...
  worldnews: 600,000 regels verwerkt, 494,748 bewaard...
  worldnews: 800,000 regels verwerkt, 662,978 bewaard...
  worldnews: 1,000,000 regels verwerkt, 825,605 bewaard...
  worldnews: 1,200,000 regels verwerkt, 989,414 bewaard...
  worldnews: 1,400,000 regels verwerkt, 1,141,505 bewaard...
  worldnews: 1,600,000 regels verwerkt, 1,299,271 bewaard...
  worldnews: 1,800,000 regels verwerkt, 1,462,455 bewaard...
  → 1,502,497 comments bewaard

Inladen: r/democrats (r_democrats_comments.jsonl)
  democrats: 200,000 regels verwerkt, 145,120 bewaard...
  → 341,541 comments bewaard

Inladen: r/republican (r_trump_comments.jsonl)
  → 117,100 comments bewaard

Inladen: r/politics (r_polit

In [4]:
# Deduplicate on id
df = df.drop_duplicates(subset=['id'])
# Remove empty/bot authors
df = df[df['author'].str.strip().ne('')]
df = df[~df['author'].isin(['[deleted]', 'AutoModerator'])]

print(f'After deduplication: {df.shape}')
print(df['subreddit'].value_counts())

Na deduplicatie: (4094357, 18)
subreddit
politics        1868477
worldnews       1498387
democrats        326938
conservative     302333
republican        98222
Name: count, dtype: int64


In [5]:
# Fix data types
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df['date'] = pd.to_datetime(df['date'])

# Export
out_path = DATA_DIR / 'reddit_clean.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
print(f'Columns: {list(df.columns)}')

Opgeslagen: ..\..\Data\Reddit\reddit_clean.parquet  (4,094,357 rijen × 18 kolommen)
Kolommen: ['id', 'parent_id', 'link_id', 'subreddit', 'created_utc', 'datetime', 'date', 'author', 'score', 'body', 'body_clean', 'text_clean', 'length_chars', 'length_words', 'sentiment', 'mentions_trump', 'mentions_harris', 'mentions_biden']


## 3. Load posts

In [6]:
def parse_posts(file_path, subreddit_name):
    """Reads a posts JSONL file and returns a list of dicts."""
    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            created_utc = obj.get('created_utc')
            if created_utc is None:
                continue

            title = obj.get('title', '')
            selftext = obj.get('selftext', '')
            if not isinstance(selftext, str):
                selftext = ''
            selftext = '' if selftext.lower().strip() in ('[deleted]', '[removed]') else selftext

            title_clean = clean_text(title)
            full_text = (title + ' ' + selftext).strip()
            full_clean = clean_text(full_text)
            sentiment = sia.polarity_scores(full_clean)['compound'] if full_clean else 0.0

            rows.append({
                'id':             obj.get('id', ''),
                'subreddit':      subreddit_name,
                'author':         obj.get('author', ''),
                'created_utc':    int(created_utc),
                'datetime':       pd.to_datetime(created_utc, unit='s', utc=True),
                'date':           pd.to_datetime(created_utc, unit='s', utc=True).date(),
                'title':          title,
                'title_clean':    title_clean,
                'selftext':       selftext,
                'score':          obj.get('score', 0),
                'upvote_ratio':   obj.get('upvote_ratio', None),
                'num_comments':   obj.get('num_comments', 0),
                'num_crossposts': obj.get('num_crossposts', 0),
                'sentiment':      sentiment,
                'mentions_trump':  int('trump' in full_clean),
                'mentions_harris': int('harris' in full_clean or 'kamala' in full_clean),
                'mentions_biden':  int('biden' in full_clean),
            })
    return rows


all_posts = []
for name, path in POST_FILES.items():
    print(f'Loading posts: r/{name}')
    posts = parse_posts(path, name)
    print(f'  → {len(posts):,} posts')
    all_posts.extend(posts)

df_posts = pd.DataFrame(all_posts)
df_posts = df_posts.drop_duplicates(subset=['id'])
df_posts['datetime'] = pd.to_datetime(df_posts['datetime'], utc=True)
df_posts['date'] = pd.to_datetime(df_posts['date'])

print(f'\nTotaal posts: {df_posts.shape}')
print(df_posts['subreddit'].value_counts())

Inladen posts: r/conservative
  → 41,790 posts
Inladen posts: r/worldnews
  → 36,854 posts
Inladen posts: r/democrats
  → 20,756 posts
Inladen posts: r/republican
  → 19,111 posts
Inladen posts: r/politics
  → 69,963 posts

Totaal posts: (188474, 17)
subreddit
politics        69963
conservative    41790
worldnews       36854
democrats       20756
republican      19111
Name: count, dtype: int64


In [7]:
out_posts = DATA_DIR / 'reddit_posts.parquet'
df_posts.to_parquet(out_posts, index=False)
print(f'Saved: {out_posts}  ({df_posts.shape[0]:,} rows × {df_posts.shape[1]} columns)')
print(f'Columns: {list(df_posts.columns)}')

Opgeslagen: ..\..\Data\Reddit\reddit_posts.parquet  (188,474 rijen × 17 kolommen)
Kolommen: ['id', 'subreddit', 'author', 'created_utc', 'datetime', 'date', 'title', 'title_clean', 'selftext', 'score', 'upvote_ratio', 'num_comments', 'num_crossposts', 'sentiment', 'mentions_trump', 'mentions_harris', 'mentions_biden']


## 4. Quick check

In [8]:
print('=== reddit_clean.parquet ===')
print(df.dtypes)
print()
print(df[['subreddit', 'date', 'sentiment', 'mentions_trump', 'mentions_harris', 'mentions_biden']].describe(include='all'))

=== reddit_clean.parquet ===
id                                str
parent_id                         str
link_id                           str
subreddit                         str
created_utc                     int64
datetime           datetime64[s, UTC]
date                    datetime64[s]
author                            str
score                           int64
body                              str
body_clean                        str
text_clean                        str
length_chars                    int64
length_words                    int64
sentiment                     float64
mentions_trump                  int64
mentions_harris                 int64
mentions_biden                  int64
dtype: object

       subreddit                 date     sentiment  mentions_trump  \
count    4094357              4094357  4.094357e+06    4.094357e+06   
unique         5                  NaN           NaN             NaN   
top     politics                  NaN           NaN        

In [9]:
print('=== reddit_posts.parquet ===')
print(df_posts.dtypes)
print()
print(df_posts[['subreddit', 'date', 'score', 'num_comments', 'sentiment']].describe())

=== reddit_posts.parquet ===
id                                str
subreddit                         str
author                            str
created_utc                     int64
datetime           datetime64[s, UTC]
date                    datetime64[s]
title                             str
title_clean                       str
selftext                          str
score                           int64
upvote_ratio                  float64
num_comments                    int64
num_crossposts                  int64
sentiment                     float64
mentions_trump                  int64
mentions_harris                 int64
mentions_biden                  int64
dtype: object

                      date          score   num_comments      sentiment
count               188474  188474.000000  188474.000000  188474.000000
mean   2024-09-01 11:40:58     416.379272      45.853608      -0.051956
min    2024-07-04 00:00:00       0.000000       0.000000      -0.999900
25%    2024-07-30 00:0